In [ ]:
from langchain_google_genai import GoogleGenerativeAI
from dotenv import load_dotenv

from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate


from pydantic import BaseModel, Field
from typing import Annotated, List, Dict

load_dotenv()

True

In [2]:
model = GoogleGenerativeAI(
    model = 'gemini-2.0-flash'
)

In [3]:
class Movie(BaseModel):
    title: Annotated[str, Field(description = 'Title of the movie')]
    director: Annotated[str, Field(description = 'Director of the movie')]
    actors: Annotated[List[str], Field(description = 'List of actors in the movie')]
    singers: Annotated[str, Field(description = 'Singers of the movie')]
    misician: Annotated[str, Field(description = 'Musician of the movie')]
    collection: Annotated[str, Field(description = 'Worldwide collection of the movie')]


parser = PydanticOutputParser(pydantic_object = Movie)

In [4]:
temp = PromptTemplate(
    template = '''Return title, director, actors, singers, musicians and world wide collection of {movie}
    {format_instructions}
    ''',
    input_variables = ['movie'],
    partial_variables = {'format_instructions': parser.get_format_instructions}
)

In [5]:
chain = temp | model | parser

response = chain.invoke(
    {'movie': 'Sholay'}
)

print(response)
print()

print(type(response))
print()

print(response.title)

title='Sholay' director='Ramesh Sippy' actors=['Dharmendra', 'Amitabh Bachchan', 'Sanjeev Kumar', 'Hema Malini', 'Jaya Bhaduri', 'Amjad Khan'] singers='Lata Mangeshkar, Kishore Kumar, Manna Dey, R. D. Burman' misician='R. D. Burman' collection='₹350 million'

<class '__main__.Movie'>

Sholay


In [6]:
for key, value in response.model_dump().items():
    print(f'{key}: {value}')

title: Sholay
director: Ramesh Sippy
actors: ['Dharmendra', 'Amitabh Bachchan', 'Sanjeev Kumar', 'Hema Malini', 'Jaya Bhaduri', 'Amjad Khan']
singers: Lata Mangeshkar, Kishore Kumar, Manna Dey, R. D. Burman
misician: R. D. Burman
collection: ₹350 million


# See what is the prompt

In [7]:


prompt = temp.invoke({'movie': 'Sholay'})

print(prompt)

text='Return title, director, actors, singers, musicians and world wide collection of Sholay\n    The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"title": {"description": "Title of the movie", "title": "Title", "type": "string"}, "director": {"description": "Director of the movie", "title": "Director", "type": "string"}, "actors": {"description": "List of actors in the movie", "items": {"type": "string"}, "title": "Actors", "type": "array"}, "singers": {"description": "Singers of the movie", "title": "Singers", "type": "string"}, "misician": {"description": "Musician 